# 04 — Avaliação, calibração e política de crédito
**Modelo:** LightGBM, conjunto A (decisões 3.2 e 3.3), target DPD > 5.
**Fluxo:** (1) calibração (ajustada na validação, confirmada no teste) → (2) métricas com IC 95%, por safra e por tipo de contrato → (3) SHAP → (4) cortes p₁ e p₂ (validação) → (5) faixas A–E → (6) simulação e perda esperada → (7) confirmação no teste → (8) aplicação na submissão e `submissao/previsoes.csv`.
Todos os cortes saem da **validação**; o **teste** só confirma.

In [ ]:
import sys, warnings; sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', module='sklearn|lightgbm|shap|pandas')  # avisos de categoria nova (train.preparar) ficam visíveis
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib, shap
from IPython.display import display, Markdown
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import lightgbm as lgb
import train as tr, policy as po
from config import GERADOS, MODELOS, OUTPUTS, SUBMISSAO, RECALL_ALVO, LGD_CENARIOS, SEED
from evaluate import metricas, limiar_max_f1, psi

pd.set_option('display.width', 220, 'display.max_columns', 60)
sns.set_theme(style='whitegrid')
FIG, TAB = OUTPUTS / 'figures', OUTPUTS / 'tables'

R = joblib.load(GERADOS / 'modelos' / 'resultado_nb03.joblib')
ALG, CONJ, modelo, cats = R['ALG'], R['CONJ_FINAL'], R['modelo'], R['cats']
b = pd.read_parquet(GERADOS / 'base_modelagem.parquet'); bs = pd.read_parquet(GERADOS / 'base_submissao_features.parquet')
sp = {s: b[b.split == s].reset_index(drop=True) for s in ['treino', 'valid', 'teste']}
Y = {s: sp[s].target_dpd5.astype(int).values for s in sp}
P = {s: R['S'][ALG, CONJ][s] for s in ['valid', 'teste']}
Xsub = tr.preparar(bs, R['features'], cats)
novas = {c: int((bs[c].notna() & ~bs[c].isin(cats[c])).sum()) for c in cats if c in bs and (bs[c].notna() & ~bs[c].isin(cats[c])).any()}
print('clientes da submissão com categoria fora do treino (viram nulo):', novas or 'nenhum')
P['sub'] = modelo.predict_proba(Xsub)[:, 1]
print(ALG, CONJ, '| AUC valid (confere com nb03):', round(roc_auc_score(Y['valid'], P['valid']), 4), '| submissão:', len(bs), 'clientes')

## 1. Calibração

A taxa de inadimplência caiu de 25,3% (treino) para 17,6% (validação) e 17,5% (teste), então o score bruto superestima o risco. Compara-se **Platt** (logística sobre o logit, monótona e sem empates) e **isotônica** (mais flexível, mas em degraus). O calibrador é ajustado na **validação** e avaliado no **teste**; a calibração é monótona e não muda a ordenação, apenas a escala em probabilidade.

In [ ]:
cal = {m: po.calibrador(m, P['valid'], Y['valid']) for m in ['platt', 'isotonica']}
rows = []
for nome, f in [('bruto', lambda p: p), *cal.items()]:
    for s in ['valid', 'teste']:
        q = f(P[s]); rows.append({'calibração': nome, 'split': s, 'brier': np.mean((q - Y[s]) ** 2), 'ECE': po.ece(Y[s], q),
                                   'score médio': q.mean(), 'taxa real': Y[s].mean(), 'AUC': roc_auc_score(Y[s], q)})
display(pd.DataFrame(rows).round(4))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, s in zip(ax, ['valid', 'teste']):
    for nome, f in [('bruto', lambda p: p), *cal.items()]:
        x, y = calibration_curve(Y[s], f(P[s]), n_bins=10, strategy='quantile')[::-1]
        a.plot(x, y, marker='o', label=nome)
    a.plot([0, .7], [0, .7], 'k--', lw=1); a.set(title=f'Confiabilidade — {s}', xlabel='probabilidade prevista', ylabel='taxa observada'); a.legend()
plt.tight_layout(); plt.savefig(FIG / 'nb04_calibracao.png', dpi=130); plt.show()

**Escolha:** a de menor Brier em validação cruzada **dentro da validação** (a tabela acima mostra o teste só como confirmação). A isotônica cria degraus (empates) que travam os cortes e as faixas; por isso ela recebe um desempate pelo score bruto (1e-6), que preserva a ordenação do modelo. A escolha é registrada em `docs/DECISOES.md`.

In [ ]:
# escolha por validação cruzada DENTRO da validação (ajusta em 4 dobras, mede o Brier na 5ª); o teste só confirma
def brier_cv(metodo):
    q = np.zeros(len(Y['valid']))
    for a, b_ in StratifiedKFold(5, shuffle=True, random_state=SEED).split(P['valid'], Y['valid']):
        q[b_] = po.calibrador(metodo, P['valid'][a], Y['valid'][a])(P['valid'][b_])
    return np.mean((q - Y['valid']) ** 2)
brier_v = {m: brier_cv(m) for m in cal}
METODO = 'platt' if brier_v['platt'] <= brier_v['isotonica'] + 1e-4 else 'isotonica'
print('Brier (validação cruzada na validação):', {k: round(v, 5) for k, v in brier_v.items()}, '->', METODO)
Q = {s: cal[METODO](P[s]) for s in P}

## 2. Métricas e robustez

In [ ]:
lim_q = limiar_max_f1(Y['valid'], Q['valid'])
N = 1000
def boot(y, q, lim):
    rng, out = np.random.default_rng(SEED), []
    for _ in range(N):
        i = rng.integers(0, len(y), len(y)); out.append(metricas(y[i], q[i], lim))
    d = pd.DataFrame(out); return d.quantile(.025), d.quantile(.975)
tab = []
for s in ['valid', 'teste']:
    m = pd.Series(metricas(Y[s], Q[s], lim_q)); lo, hi = boot(Y[s], Q[s], lim_q)
    tab.append(pd.DataFrame({'valor': m, 'ic95_inf': lo, 'ic95_sup': hi}).assign(split=s))
tab = pd.concat(tab).reset_index(names='métrica'); tab.to_csv(TAB / 'nb04_metricas_ic.csv', index=False)
print('limiar de máximo F1 (probabilidade calibrada):', round(lim_q, 4)); display(tab.pivot(index='métrica', columns='split').round(4))

In [ ]:
d = pd.concat([sp['valid'].assign(q=Q['valid'], y=Y['valid']), sp['teste'].assign(q=Q['teste'], y=Y['teste'])])
def seg(g):
    return pd.Series({'n': len(g), 'taxa real': g.y.mean(), 'score médio': g.q.mean(),
                      'AUC': roc_auc_score(g.y, g.q) if g.y.nunique() > 1 else np.nan, 'recall (lim F1)': (g.q[g.y == 1] >= lim_q).mean()})
por_safra = d.groupby(['split', d.safra.astype(str).str[:4]]).apply(seg, include_groups=False); por_tipo = d.groupby(['split', 'tipo_contrato']).apply(seg, include_groups=False)
display(por_safra.round(3)); display(por_tipo.round(3))
por_safra.round(4).to_csv(TAB / 'nb04_por_safra.csv'); por_tipo.round(4).to_csv(TAB / 'nb04_por_tipo_contrato.csv')
print('PSI do score calibrado — valid × teste:', round(psi(Q['valid'], Q['teste']), 4), '| teste × submissão:', round(psi(Q['teste'], Q['sub']), 4))

## 3. Explicabilidade (SHAP)

In [ ]:
Xv = tr.preparar(sp['valid'], R['features'], cats).sample(5000, random_state=SEED)
sv = shap.TreeExplainer(modelo)(Xv)
vals = sv.values[:, :, 1] if sv.values.ndim == 3 else sv.values
imp = pd.Series(np.abs(vals).mean(0), index=Xv.columns).sort_values(ascending=False)
display(imp.round(4).to_frame('|SHAP| médio (log-odds)'))
shap.summary_plot(vals, Xv, show=False, max_display=15); plt.tight_layout(); plt.savefig(FIG / 'nb04_shap_global.png', dpi=130); plt.show()

In [ ]:
# local: um cliente de risco alto e um de risco baixo da amostra
pq = pd.Series(modelo.predict_proba(Xv)[:, 1], index=Xv.index)
for rotulo, i in [('risco alto', pq.idxmax()), ('risco baixo', pq.idxmin())]:
    k = Xv.index.get_loc(i); top = pd.Series(vals[k], index=Xv.columns).reindex(imp.index[:6])
    print(f'{rotulo}: score {pq[i]:.3f}'); display(pd.DataFrame({'valor': Xv.loc[i, top.index].astype(str), 'contribuição (log-odds)': top.round(3)}))

## 4. Cortes p₁ e p₂ (escolhidos na validação)
- **p₁** (fronteira do *aprovar direto*): maior corte para o qual a zona "não aprovar direto" (ajuste + rejeição) captura **≥ 80% dos inadimplentes** (recall-alvo, `config.RECALL_ALVO`).
- **p₂** (fronteira do *rejeitar*): corte de **máximo F1** entre os contratos acima de p₁.
- Entre p₁ e p₂: **aprovar com ajuste**.

In [ ]:
p1 = po.corte_p1(Y['valid'], Q['valid'], RECALL_ALVO); p2 = po.corte_p2(Y['valid'], Q['valid'], p1)
res = pd.DataFrame({s: po.resumo_corte(Y[s], Q[s], p1, p2) for s in ['valid', 'teste']})
print(f'p1 = {p1:.4f}  |  p2 = {p2:.4f}'); display(res.round(4))

In [ ]:
# curva (p1, p2): taxa de aprovação direta × inadimplência dos aprovados × recall × precision
g = []
for t1 in np.linspace(.05, .35, 31):
    r = po.resumo_corte(Y['valid'], Q['valid'], t1, max(t1, p2)); g.append({'p1': t1, **r})
g = pd.DataFrame(g); g.to_csv(TAB / 'nb04_curva_p1.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(g.taxa_aprovacao_direta, g.recall_nao_aprovar, marker='o'); ax[0].axhline(RECALL_ALVO, ls='--', c='gray')
ax[0].set(xlabel='taxa de aprovação direta', ylabel='recall (barrar inadimplente)', title='Recall × aprovação (validação)')
ax[1].plot(g.taxa_aprovacao_direta, g.inad_aprovados_diretos, marker='o')
ax[1].set(xlabel='taxa de aprovação direta', ylabel='inadimplência dos aprovados diretos', title='Risco da carteira aprovada')
plt.tight_layout(); plt.savefig(FIG / 'nb04_tradeoff_p1.png', dpi=130); plt.show()
display(g.iloc[::3][['p1', 'taxa_aprovacao_direta', 'inad_aprovados_diretos', 'recall_nao_aprovar', 'precision_nao_aprovar', 'bons_barrados_por_mau_barrado']].round(3))

## 5. Faixas A–E (visão de gestão)
**A decisão é tomada só por p₁ e p₂ (3 decisões).** As faixas A–E servem apenas para acompanhamento: A/B dividem a zona "aprovar" e C/D a zona "ajuste" na mediana do volume da validação; E é a rejeição. Verifica-se monotonicidade e o IC de Wilson entre faixas vizinhas.
**Resultado:** as taxas crescem de A a E, mas os ICs de A, B, C (e D no teste) se sobrepõem: com AUC ≈ 0,70 o modelo **não distingue estatisticamente as faixas A–D**; só E se separa. Por isso as faixas não sustentam decisões próprias.

In [ ]:
cf = po.cortes_faixas(Q['valid'], p1, p2); print('cortes das faixas:', np.round(cf, 4))
tf = {s: po.tabela_faixas(Y[s], Q[s], cf) for s in ['valid', 'teste']}
for s in tf: print(s); display(tf[s].round(4))
for s in tf: print(s, '| monotônica:', bool(tf[s].taxa.is_monotonic_increasing), '| faixas vizinhas com IC sobreposto:', tf[s].sobrepoe_proxima[:-1].sum())
pd.concat(tf, names=['split']).round(4).to_csv(TAB / 'nb04_faixas.csv')

## 6. Simulação e perda esperada
**EL = PD × LGD × EAD**, com PD calibrada, EAD = `valor_credito` e **LGD como premissa** (45%, 60%, 75%): os dados não trazem recuperação. Compara-se a carteira sem política (aprovar tudo) com a política: aprovar direto (A/B), aprovar com ajuste (C/D, exposição **sem** redução: o efeito do ajuste não é medido) e rejeitar (E, exposição zero).

In [ ]:
sim = []
for s in ['valid', 'teste']:
    ead, q = sp[s].valor_credito.values, Q[s]; y = Y[s]
    for lgd in LGD_CENARIOS:
        el_tudo = po.perda_esperada(q, ead, lgd); el_pol = po.perda_esperada(q, ead, lgd, q < p2)
        el_dir = po.perda_esperada(q, ead, lgd, q < p1)
        perda_real = ((y * ead)[q < p2].sum()) * lgd
        sim.append({'split': s, 'LGD': lgd, 'EL aprovar tudo': el_tudo, 'EL com política (A–D)': el_pol, 'EL só aprovados diretos (A/B)': el_dir,
                    'redução da EL (política)': 1 - el_pol / el_tudo, 'perda observada (A–D)': perda_real,
                    '% do crédito rejeitado': ead[q >= p2].sum() / ead.sum()})
sim = pd.DataFrame(sim); sim.to_csv(TAB / 'nb04_simulacao_el.csv', index=False)
display(sim.round(3))

## 7. Confirmação no teste
As métricas do par (p₁, p₂) no teste devem ficar dentro do IC 95% da validação (bootstrap).

In [ ]:
def boot_pol(s):
    rng, out = np.random.default_rng(SEED), []; y, q = Y[s], Q[s]
    for _ in range(N):
        i = rng.integers(0, len(y), len(y)); out.append(po.resumo_corte(y[i], q[i], p1, p2))
    d = pd.DataFrame(out); return d.quantile(.025), d.quantile(.975)
lo, hi = boot_pol('valid')
chk = pd.DataFrame({'validação': res.valid, 'IC inf': lo, 'IC sup': hi, 'teste': res.teste})
chk['teste dentro do IC'] = (chk.teste >= chk['IC inf']) & (chk.teste <= chk['IC sup'])
display(chk.round(4)); chk.round(4).to_csv(TAB / 'nb04_confirmacao_teste.csv')

### Clientes já vistos no treino × clientes novos
Muitos contratos de validação e teste são de clientes que já aparecem no treino (com outros contratos). Como as variáveis são de cadastro, o modelo pode "reconhecer" o cliente. Comparam-se os dois grupos com o mesmo par (p₁, p₂).

In [ ]:
vistos = set(sp['treino'].id_cliente); lin = {}
for s in ['valid', 'teste']:
    visto = sp[s].id_cliente.isin(vistos).values
    for nome, m in [('cliente já visto', visto), ('cliente novo', ~visto)]:
        r = po.resumo_corte(Y[s][m], Q[s][m], p1, p2)
        lin[s, nome] = {'n': int(m.sum()), 'taxa real': Y[s][m].mean(), 'AUC': roc_auc_score(Y[s][m], Q[s][m]),
                        'aprovação direta': r['taxa_aprovacao_direta'], 'recall (não aprovar direto)': r['recall_nao_aprovar'], 'rejeição': r['taxa_rejeicao']}
vn = pd.DataFrame(lin).T; display(vn.round(4)); vn.round(4).to_csv(TAB / 'nb04_clientes_vistos_novos.csv')
print('clientes da submissão que já aparecem no treino:', round(bs.id_cliente.isin(vistos).mean(), 3))

## 8. Fichas de decisão

In [ ]:
def ficha(nome, o_que, criterio, linhas):
    t = f'**{nome}**\n\n1. **O que foi decidido:** {o_que}\n2. **Critério:** {criterio}\n'
    return t + '\n'.join(linhas)
rv, rt = res.valid, res.teste
alvo_ok = f"recall de 'não aprovar direto' = {rv.recall_nao_aprovar:.1%} (validação) e {rt.recall_nao_aprovar:.1%} (teste)"
display(Markdown(ficha('p₁ — fronteira do aprovar direto', f'p₁ = {p1:.3f} (probabilidade calibrada)', f'menor risco aceito em aprovação direta com {RECALL_ALVO:.0%} de recall-alvo',
 [f"3. **Números:** aprova direto {rv.taxa_aprovacao_direta:.1%} (valid) / {rt.taxa_aprovacao_direta:.1%} (teste); inadimplência dos aprovados diretos {rv.inad_aprovados_diretos:.1%} / {rt.inad_aprovados_diretos:.1%}; {alvo_ok}.",
  f"4. **Alternativas:** a curva da seção 4 mostra o custo de cada p₁ vizinho em aprovação e recall.",
  f"5. **Trade-off:** {rv.bons_barrados_por_mau_barrado:.1f} bons clientes barrados do aprovar-direto para cada inadimplente (validação).",
  f"6. **Impacto:** ver seção 6 (EL sob LGD de 45/60/75%).",
  "7. **Riscos:** deriva da taxa de inadimplência (25,3% → 17,5%), viés de seleção (só aprovados), efeito dos ajustes não medido; revisar a cada safra nova com PSI > 0,10."])))
display(Markdown(ficha('p₂ — fronteira do rejeitar', f'p₂ = {p2:.3f}', 'máximo F1 entre os contratos acima de p₁',
 [f"3. **Números:** rejeita {rv.taxa_rejeicao:.1%} (valid) / {rt.taxa_rejeicao:.1%} (teste); inadimplência dos rejeitados {rv.inad_rejeitados:.1%} / {rt.inad_rejeitados:.1%}; precision {rv.precision_rejeitar:.1%} / {rt.precision_rejeitar:.1%}; recall {rv.recall_rejeitar:.1%} / {rt.recall_rejeitar:.1%}.",
  "4. **Alternativas:** p₂ maior rejeita menos e com mais certeza; p₂ menor barra mais maus, mas recusa mais bons clientes.",
  f"5. **Trade-off:** inadimplência dos rejeitados é {rv.inad_rejeitados / Y['valid'].mean():.1f}x a média da base.",
  "6. **Impacto:** ver seção 6.", "7. **Riscos:** os rejeitados deixam de gerar desfecho observável (reject inference fora do escopo)."])))

## 9. Aplicação na submissão

In [ ]:
sub = pd.DataFrame({'id_cliente': bs.id_cliente.values, 'probabilidade': Q['sub']})
sub['faixa_gestao'] = po.faixa(sub.probabilidade, cf).values; sub['decisao'] = sub.faixa_gestao.map(po.DECISAO)
assert sub.id_cliente.is_unique and len(sub) == bs.id_cliente.nunique()
SUBMISSAO.mkdir(exist_ok=True); sub[['id_cliente', 'probabilidade', 'decisao', 'faixa_gestao']].to_csv(SUBMISSAO / 'previsoes.csv', index=False)
dec = pd.DataFrame({'submissão': sub.decisao.value_counts(normalize=True),
                    'teste': po.faixa(Q['teste'], cf).map(po.DECISAO).value_counts(normalize=True),
                    'validação': po.faixa(Q['valid'], cf).map(po.DECISAO).value_counts(normalize=True)})
display(dec.round(4))
print('PSI do score (teste × submissão):', round(psi(Q['teste'], Q['sub']), 4), '| probabilidade média submissão × teste:', round(sub.probabilidade.mean(), 4), '×', round(Q['teste'].mean(), 4))

## 10. Por que a submissão parece mais arriscada?
Três verificações, todas na escala das variáveis do modelo (teste × submissão):
1. **PSI por variável:** alguma variável mudou de distribuição?
2. **Diferença média de contribuição SHAP** (submissão − teste, em log-odds): quais variáveis empurram o risco para cima?
3. **Validação adversarial:** um classificador tenta distinguir teste de submissão; AUC 0,5 = indistinguíveis, perto de 1 = populações bem diferentes.

In [ ]:
te, F = sp['teste'], R['features']
Xt = tr.preparar(te, F, cats).sample(5000, random_state=SEED); Xs = Xsub.sample(5000, random_state=SEED)
ex = shap.TreeExplainer(modelo)
contrib = lambda X: (lambda v: v[:, :, 1] if v.ndim == 3 else v)(ex(X).values)
desvio = pd.Series(contrib(Xs).mean(0) - contrib(Xt).mean(0), index=F)
shift = pd.DataFrame({'PSI': {f: psi(te[f], bs[f]) for f in F}, 'Δ SHAP médio (log-odds)': desvio}).sort_values('Δ SHAP médio (log-odds)', key=abs, ascending=False)
display(shift.round(4)); shift.round(4).to_csv(TAB / 'nb04_shift_submissao.csv')
print('soma das diferenças de contribuição (log-odds):', round(desvio.sum(), 3))
Xadv = pd.concat([tr.preparar(te, F, cats), Xsub]); yadv = np.r_[np.zeros(len(te)), np.ones(len(bs))]
padv = cross_val_predict(lgb.LGBMClassifier(n_estimators=200, learning_rate=.05, verbose=-1, random_state=SEED), Xadv, yadv,
                         cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
print('AUC adversarial (teste × submissão):', round(roc_auc_score(yadv, padv), 3))

## 11. Modelo final
O modelo, o calibrador e os cortes ficam juntos em `models/modelo_final.joblib`, para pontuar novos clientes sem rodar os notebooks (uso no README).

In [ ]:
import importlib.metadata as im
MODELOS.mkdir(exist_ok=True)
joblib.dump({'versoes': {p: im.version(p) for p in ['lightgbm', 'scikit-learn', 'pandas', 'numpy']},
             'algoritmo': ALG, 'conjunto': CONJ, 'modelo': modelo, 'calibrador': cal[METODO], 'metodo_calibracao': METODO,
             'features': R['features'], 'categorias': cats, 'p1': p1, 'p2': p2, 'cortes_faixas': cf,
             'hiperparametros': R['T'][ALG, CONJ]['params'], 'n_arvores': R['T'][ALG, CONJ]['n_arvores'],
             'metricas': tab.pivot(index='métrica', columns='split', values='valor').loc[['auc', 'f1', 'recall', 'brier']].round(4).to_dict()},
            MODELOS / 'modelo_final.joblib')
print('salvo:', MODELOS / 'modelo_final.joblib', round((MODELOS / 'modelo_final.joblib').stat().st_size / 1e6, 1), 'MB')

### Leitura
- **Calibração:** o score bruto superestima o risco (média 0,22 contra 17,5% observado). A isotônica, com desempate pelo score bruto, calibra melhor (Brier 0,1181 contra 0,1208 da Platt no teste).
- **Desempenho:** AUC 0,69–0,70 (IC 95% de ±0,012), estável entre 2023 e 2024 e entre os tipos de contrato (crédito rotativo é o pior, AUC ≈ 0,68).
- **Política (3 decisões):** p₁ = 0,118 e p₂ = 0,250. Na validação, 33% são aprovados direto (inadimplência 10,5%), 56% aprovados com ajuste (13,1%) e 11% rejeitados (62,9%). O recall de "não aprovar direto" é 80,0% na validação e 78,9% no teste (dentro do IC). Custo: ≈ 3,7 bons clientes barrados por inadimplente.
- **Faixas A–E:** só visão de gestão; A–D não se distinguem estatisticamente (ICs de Wilson sobrepostos), apenas E se separa.
- **Submissão:** PSI do score 0,067 (< 0,25), mas rejeição de 19,9% contra 11,0% no teste e probabilidade média de 0,225 contra 0,176. Nenhuma variável isolada mudou (PSI máximo 0,04); o risco maior vem de pequenos deslocamentos em várias variáveis (soma +0,20 log-odds, sobretudo renda, organização e ocupação), e o AUC adversarial de 0,61 indica uma diferença moderada de perfil.
  Não dá para separar mudança real de perfil de efeito de seleção (o modelo só viu contratos aprovados); a taxa de rejeição da submissão é uma estimativa que supõe que a relação perfil → risco se mantém. **Alerta registrado.**
- **Limites:** a perda esperada depende da LGD (premissa) e o efeito dos ajustes não foi medido.